In [1]:
# Generate token embeddings from the model(s)
import pandas as pd
import numpy as np
from tqdm import tqdm
from rdkit import Chem
from transformers import AutoModel, AutoTokenizer
import torch


In [2]:
# load model from huggingface
def load_model(model_name="aaronfeller/PeptideMLM-MTR_lg"):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
    return model, tokenizer

def generate_embeddings(model, tokenizer, sequences, batch_size=64):
    embeddings = []
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch_seq = sequences[i:i+batch_size]
        inputs = tokenizer(batch_seq, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        # mean pool the token embeddings to get a single embedding per sequence
        batch_embeddings = outputs['mean_pool'].cpu().numpy()
        embeddings.extend(batch_embeddings)
    return np.array(embeddings)


In [10]:
from sklearn.metrics import matthews_corrcoef

# train on THPep data 
df_THPep = pd.read_csv('finetuning_data/THPep/THPep_main90_smiles_classes.csv')


for model_name in ["aaronfeller/PeptideMLM_lg", "aaronfeller/PeptideMLM-MTR_lg", "aaronfeller/PeptideMTR_lg"]:
    print(f'Training Random Forest Classifer with embeddings from model: {model_name}')
    model, tokenizer = load_model(model_name)

    # generate embeddings for all sequences in the dataset
    all_embeddings = generate_embeddings(model, tokenizer, df_THPep['SMILES'].tolist())

    # store embeddings in dataframe
    df_THPep_embeddings = pd.DataFrame(all_embeddings)
    df_THPep_embeddings['label'] = df_THPep['Class'].tolist()

    MCC_scores = []
    for iteration in range(20):
        # print(f'Iteration {iteration+1}/20')

        train_df, test_df = train_test_split(df_THPep_embeddings, test_size=0.2, random_state=(iteration+1)*52984)

        train_embeddings = train_df.drop('label', axis=1).values
        train_labels = train_df['label'].values

        test_embeddings = test_df.drop('label', axis=1).values
        test_labels = test_df['label'].values

        rf_reg = RandomForestClassifier(random_state=(iteration+1)*43872, n_estimators=100, n_jobs=100)
        rf_reg.fit(train_embeddings, train_labels)

        test_predictions = rf_reg.predict(test_embeddings)

        # calculate MCC score
        f1 = classification_report(test_labels, test_predictions, output_dict=True)['weighted avg']['f1-score']
        mcc = matthews_corrcoef(test_labels, test_predictions)
        MCC_scores.append(mcc)
        # print(f'MCC score for iteration {iteration+1}: {mcc}')

    # print average MCC score across iterations
    print(f'Average MCC score across iterations: {np.mean(MCC_scores)}')

Training Random Forest Classifer with embeddings from model: aaronfeller/PeptideMLM_lg


100%|██████████| 10/10 [00:06<00:00,  1.66it/s]


Average MCC score across iterations: 0.6017624574629367
Training Random Forest Classifer with embeddings from model: aaronfeller/PeptideMLM-MTR_lg


100%|██████████| 10/10 [00:05<00:00,  1.91it/s]


Average MCC score across iterations: 0.636874697951081
Training Random Forest Classifer with embeddings from model: aaronfeller/PeptideMTR_lg


100%|██████████| 10/10 [00:06<00:00,  1.54it/s]


Average MCC score across iterations: 0.6381514648353825


In [11]:
from sklearn.metrics import matthews_corrcoef

# use train and test splits from CellPPD-MOD
df_CellPPD_train = pd.read_csv('finetuning_data/CellPPD-MOD/CellPPD_training_data/CellPPD_train.csv')
df_CellPPD_test = pd.read_csv('finetuning_data/CellPPD-MOD/CellPPD_training_data/CellPPD_test.csv')


for i, model_name in enumerate(["aaronfeller/PeptideMLM_lg", "aaronfeller/PeptideMLM-MTR_lg", "aaronfeller/PeptideMTR_lg"]):
    print(f'Training Random Forest Classifer with embeddings from model: {model_name}')
    model, tokenizer = load_model(model_name)

    # generate embeddings for all sequences in the dataset
    train_embeddings = generate_embeddings(model, tokenizer, df_CellPPD_train['smiles'].tolist())
    test_embeddings = generate_embeddings(model, tokenizer, df_CellPPD_test['smiles'].tolist())

    # store embeddings in dataframe
    df_CellPPD_train_embeddings = pd.DataFrame(train_embeddings)
    df_CellPPD_train_embeddings['label'] = df_CellPPD_train['label'].tolist()

    df_CellPPD_test_embeddings = pd.DataFrame(test_embeddings)
    df_CellPPD_test_embeddings['label'] = df_CellPPD_test['label'].tolist()

    train_embeddings = df_CellPPD_train_embeddings.drop('label', axis=1).values
    train_labels = df_CellPPD_train_embeddings['label'].values

    test_embeddings = df_CellPPD_test_embeddings.drop('label', axis=1).values
    test_labels = df_CellPPD_test_embeddings['label'].values

    mcc_scores = []
    for iteration in range(20):
        # print(f'Iteration {iteration+1}/3')

        rf_reg = RandomForestClassifier(random_state=(iteration+1)*21428, n_estimators=100, n_jobs=100)
        rf_reg.fit(train_embeddings, train_labels)

        test_predictions = rf_reg.predict(test_embeddings)

        # calculate MCC score
        f1 = classification_report(test_labels, test_predictions, output_dict=True)['weighted avg']['f1-score']
        mcc = matthews_corrcoef(test_labels, test_predictions)
        mcc_scores.append(mcc)
        # print(f'MCC score for iteration {iteration+1}: {mcc}')

    # print average MCC score across iterations
    print(f'Average MCC score across iterations: {np.mean(mcc_scores)}')


Training Random Forest Classifer with embeddings from model: aaronfeller/PeptideMLM_lg


100%|██████████| 5/5 [00:05<00:00,  1.08s/it]


Average MCC score across iterations: 0.7552054306038297
Training Random Forest Classifer with embeddings from model: aaronfeller/PeptideMLM-MTR_lg


100%|██████████| 5/5 [00:04<00:00,  1.09it/s]


Average MCC score across iterations: 0.7807720843464863
Training Random Forest Classifer with embeddings from model: aaronfeller/PeptideMTR_lg


100%|██████████| 5/5 [00:05<00:00,  1.02s/it]


Average MCC score across iterations: 0.7807211613582219


In [12]:
from sklearn.metrics import matthews_corrcoef

# load AmpHGT train and test splits
df_AmpHGT_test = pd.read_csv('finetuning_data/AmpHGT/datasets_ncaa/amp_test.csv')
df_AmpHGT_train = pd.read_csv('finetuning_data/AmpHGT/datasets_ncaa/amp_train.csv')

for i, model_name in enumerate(["aaronfeller/PeptideMLM_lg", "aaronfeller/PeptideMLM-MTR_lg", "aaronfeller/PeptideMTR_lg"]):
    print(f'Training Random Forest Classifer with embeddings from model: {model_name}')
    model, tokenizer = load_model(model_name)

    # generate embeddings for all sequences in the dataset
    train_embeddings = generate_embeddings(model, tokenizer, df_AmpHGT_train['smiles'].tolist())
    test_embeddings = generate_embeddings(model, tokenizer, df_AmpHGT_test['smiles'].tolist())

    # store embeddings in dataframe
    df_AmpHGT_train_embeddings = pd.DataFrame(train_embeddings)
    df_AmpHGT_train_embeddings['label'] = df_AmpHGT_train['label'].tolist()

    df_AmpHGT_test_embeddings = pd.DataFrame(test_embeddings)
    df_AmpHGT_test_embeddings['label'] = df_AmpHGT_test['label'].tolist()

    train_embeddings = df_AmpHGT_train_embeddings.drop('label', axis=1).values
    train_labels = df_AmpHGT_train_embeddings['label'].values

    test_embeddings = df_AmpHGT_test_embeddings.drop('label', axis=1).values
    test_labels = df_AmpHGT_test_embeddings['label'].values

    mcc_scores = []
    for iteration in range(20):
        # print(f'Iteration {iteration+1}/20')

        rf_reg = RandomForestClassifier(random_state=(iteration+1)*43872, n_estimators=100, n_jobs=100)
        rf_reg.fit(train_embeddings, train_labels)

        test_predictions = rf_reg.predict(test_embeddings)

        # calculate MCC score
        f1 = classification_report(test_labels, test_predictions, output_dict=True)['weighted avg']['f1-score']
        mcc = matthews_corrcoef(test_labels, test_predictions)
        mcc_scores.append(mcc)
        # print(f'MCC score for iteration {iteration+1}: {mcc}')

    # print average MCC score across iterations
    print(f'Average MCC score across iterations: {np.mean(mcc_scores)}')

Training Random Forest Classifer with embeddings from model: aaronfeller/PeptideMLM_lg


100%|██████████| 81/81 [01:46<00:00,  1.32s/it]


Average MCC score across iterations: 0.6615742650936811
Training Random Forest Classifer with embeddings from model: aaronfeller/PeptideMLM-MTR_lg


100%|██████████| 81/81 [01:46<00:00,  1.31s/it]


Average MCC score across iterations: 0.618080132804929
Training Random Forest Classifer with embeddings from model: aaronfeller/PeptideMTR_lg


100%|██████████| 81/81 [01:42<00:00,  1.27s/it]


Average MCC score across iterations: 0.488766021121667


In [14]:
from sklearn.metrics import matthews_corrcoef
from rdkit import Chem
from tqdm import tqdm

def generate_mfps(smiles_list):
    mfps = []
    for smiles in tqdm(smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            mfp = Chem.rdmolops.RDKFingerprint(mol)
            # convert bit vector to numpy array
            mfp = np.array(list(map(int, mfp.ToBitString())))
            mfps.append(mfp)
        else:
            mfps.append(None)
    return mfps

#################
### RUN THPep ###
#################
print('Running THPep with RDKit fingerprints')

# train on THPep data 
df_THPep = pd.read_csv('finetuning_data/THPep/THPep_main90_smiles_classes.csv')

# generate embeddings for all sequences in the dataset
all_embeddings = generate_mfps(df_THPep['SMILES'].tolist())

# store embeddings in dataframe
df_THPep_embeddings = pd.DataFrame(all_embeddings)
df_THPep_embeddings['label'] = df_THPep['Class'].tolist()

MCC_scores = []
for iteration in range(20):
    # print(f'Iteration {iteration+1}/20')

    train_df, test_df = train_test_split(df_THPep_embeddings, test_size=0.2, random_state=(iteration+1)*45671)

    train_embeddings = train_df.drop('label', axis=1).values
    train_labels = train_df['label'].values

    test_embeddings = test_df.drop('label', axis=1).values
    test_labels = test_df['label'].values

    rf_reg = RandomForestClassifier(random_state=(iteration+1)*43872, n_estimators=100, n_jobs=100)
    rf_reg.fit(train_embeddings, train_labels)

    test_predictions = rf_reg.predict(test_embeddings)

    # calculate MCC score
    f1 = classification_report(test_labels, test_predictions, output_dict=True)['weighted avg']['f1-score']
    mcc = matthews_corrcoef(test_labels, test_predictions)
    MCC_scores.append(mcc)
    # print(f'MCC score for iteration {iteration+1}: {mcc}')

# print average MCC score across iterations
print(f'Average MCC score across iterations: {np.mean(MCC_scores)}')

###################
### RUN CellPPD ###
###################
print('Running CellPPD-MOD with RDKit fingerprints')

# use train and test splits from CellPPD-MOD
df_CellPPD_train = pd.read_csv('finetuning_data/CellPPD-MOD/CellPPD_training_data/CellPPD_train.csv')
df_CellPPD_test = pd.read_csv('finetuning_data/CellPPD-MOD/CellPPD_training_data/CellPPD_test.csv')


# generate embeddings for all sequences in the dataset
train_embeddings = generate_mfps(df_CellPPD_train['smiles'].tolist())
test_embeddings = generate_mfps(df_CellPPD_test['smiles'].tolist())

# store embeddings in dataframe
df_CellPPD_train_embeddings = pd.DataFrame(train_embeddings)
df_CellPPD_train_embeddings['label'] = df_CellPPD_train['label'].tolist()

df_CellPPD_test_embeddings = pd.DataFrame(test_embeddings)
df_CellPPD_test_embeddings['label'] = df_CellPPD_test['label'].tolist()

train_embeddings = df_CellPPD_train_embeddings.drop('label', axis=1).values
train_labels = df_CellPPD_train_embeddings['label'].values

test_embeddings = df_CellPPD_test_embeddings.drop('label', axis=1).values
test_labels = df_CellPPD_test_embeddings['label'].values

mcc_scores = []
for iteration in range(20):
    # print(f'Iteration {iteration+1}/3')

    rf_reg = RandomForestClassifier(random_state=(iteration+1)*87236, n_estimators=100, n_jobs=100)
    rf_reg.fit(train_embeddings, train_labels)

    test_predictions = rf_reg.predict(test_embeddings)

    # calculate MCC score
    f1 = classification_report(test_labels, test_predictions, output_dict=True)['weighted avg']['f1-score']
    mcc = matthews_corrcoef(test_labels, test_predictions)
    mcc_scores.append(mcc)
    # print(f'MCC score for iteration {iteration+1}: {mcc}')

# print average MCC score across iterations
print(f'Average MCC score across iterations: {np.mean(mcc_scores)}')


##################
### RUN AmpHGT ###
##################
print('Running AmpHGT with RDKit fingerprints')

# load AmpHGT train and test splits
df_AmpHGT_test = pd.read_csv('finetuning_data/AmpHGT/datasets_ncaa/amp_test.csv')
df_AmpHGT_train = pd.read_csv('finetuning_data/AmpHGT/datasets_ncaa/amp_train.csv')


# generate embeddings for all sequences in the dataset
train_embeddings = generate_mfps(df_AmpHGT_train['smiles'].tolist())
test_embeddings = generate_mfps(df_AmpHGT_test['smiles'].tolist())

# store embeddings in dataframe
df_AmpHGT_train_embeddings = pd.DataFrame(train_embeddings)
df_AmpHGT_train_embeddings['label'] = df_AmpHGT_train['label'].tolist()

df_AmpHGT_test_embeddings = pd.DataFrame(test_embeddings)
df_AmpHGT_test_embeddings['label'] = df_AmpHGT_test['label'].tolist()

train_embeddings = df_AmpHGT_train_embeddings.drop('label', axis=1).values
train_labels = df_AmpHGT_train_embeddings['label'].values

test_embeddings = df_AmpHGT_test_embeddings.drop('label', axis=1).values
test_labels = df_AmpHGT_test_embeddings['label'].values

mcc_scores = []
for iteration in range(20):
    # print(f'Iteration {iteration+1}/20')

    rf_reg = RandomForestClassifier(random_state=(iteration+1)*63985, n_estimators=100, n_jobs=100)
    rf_reg.fit(train_embeddings, train_labels)

    test_predictions = rf_reg.predict(test_embeddings)

    # calculate MCC score
    f1 = classification_report(test_labels, test_predictions, output_dict=True)['weighted avg']['f1-score']
    mcc = matthews_corrcoef(test_labels, test_predictions)
    mcc_scores.append(mcc)
    # print(f'MCC score for iteration {iteration+1}: {mcc}')

# print average MCC score across iterations
print(f'Average MCC score across iterations: {np.mean(mcc_scores)}')

Running THPep with RDKit fingerprints


100%|██████████| 609/609 [00:01<00:00, 327.09it/s]


Average MCC score across iterations: 0.4777614351073047
Running CellPPD-MOD with RDKit fingerprints


100%|██████████| 300/300 [00:00<00:00, 319.06it/s]


Average MCC score across iterations: 0.7715770974167122
Running AmpHGT with RDKit fingerprints


100%|██████████| 5148/5148 [00:31<00:00, 163.79it/s]


Average MCC score across iterations: 0.8130610276195327
